<a href="https://colab.research.google.com/github/mcjkurz/qhchina/blob/main/tutorials/Intro_to_Python_for_Chinese_Humanities_Part_4_Network_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to networkx

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random
import pandas as pd

!pip install git+https://github.com/Hsins/mpl-tc-fonts.git

import mpl_tc_fonts
mpl_tc_fonts.load_font('cwtex', 'copy')
mpl_tc_fonts.set_font('Noto Sans CJK TC')

!pip install pyvis
!pip3 install python-louvain

In [ ]:
G = nx.Graph()

my_nodes = ["小王", "小李", "小张", "小刘", "小吴", "小陈"]

G.add_nodes_from(my_nodes)

my_edges = [("小王", "小张"), ("小李", "小刘"), ("小王", "小李"), ("小刘", "小王"), ("小张", "小李"), ("小陈", "小李"), ("小张", "小吴")]

G.add_edges_from(my_edges)

plt.figure()
nx.draw(G,with_labels=True, node_color="lightblue", node_size=1000)
plt.show()

In [ ]:
import pandas as pd

# Create a DataFrame for nodes
nodes_df = pd.DataFrame(my_nodes, columns=["Id"])

# Create a DataFrame for edges
edges_df = pd.DataFrame(my_edges, columns=["Source", "Target"])

# Save nodes and edges to CSV files
nodes_df.to_csv("nodes_small.csv", index=False, encoding='utf-8')
edges_df.to_csv("edges_small.csv", index=False, encoding='utf-8')

print("Files 'nodes_small.csv' and 'edges_smalla.csv' have been saved.")

In [ ]:
G.degree("小刘")

In [ ]:
# Calculate and display the degree of each node
print("Node Degrees:")
for node in G.nodes():
    print(f"{node}: {G.degree(node)}")

In [ ]:
# Calculate degree centrality
dc = nx.degree_centrality(G)
print("Degree Centrality:")
for node, centrality in dc.items():
    print(f"{node}: {centrality}")

In [ ]:
# Calculate betweenness centrality
bc = nx.betweenness_centrality(G)
print("Betweenness Centrality:")
for node, centrality in bc.items():
    print(f"{node}: {centrality}")

In [ ]:
# Calculate closeness centrality
cc = nx.closeness_centrality(G)
print("Closeness Centrality:")
for node, centrality in cc.items():
    print(f"{node}: {centrality}")

In [ ]:
# Find the node with the highest degree centrality
most_important_node = max(dc.items(), key = lambda x:x[1])[0] # We use [0] because .items() returns a list of tuples (name, centrality), and we need the name

print(most_important_node)

# Draw the graph, highlighting the most important node
plt.figure()
nx.draw(
    G,
    with_labels=True,
    node_color=["lightblue" if node != most_important_node else "orange" for node in G.nodes()],
    node_size=1000
)
plt.title("Graph with Highlighted Node (Highest Degree Centrality)")
plt.show()

# Historical Network Analysis

In [ ]:
!wget https://github.com/mcjkurz/qhchina/raw/refs/heads/main/tutorials/data/cbdb-ming-letters-export.csv

In [ ]:
import pandas as pd

# Load the CSV file
file_path = 'cbdb-ming-letters-export.csv'  # Replace with the path to your CSV file
data = pd.read_csv(file_path)

data

In [ ]:
# Define a reusable condition for filtering
def is_valid(value):
    if not isinstance(value, str):
      return False
    value = value.strip()
    if value in ['cd', 'se', '']:
      return False
    if pd.isna(value):
      return False
    return True

# Apply the filtering condition once and extract the receiver
data['receiver'] = data['assoc_person'].str.extract(r'^(.*)\s\(\d+\)$')[0]

# Filter out rows where writer or receiver are not valid
clean_data = data[data['writer'].apply(is_valid) & data['receiver'].apply(is_valid)]

# Select only the writer and receiver columns

connections = clean_data[['writer', 'receiver']]

connections['weight'] = 1  # Initialize a weight column with value 1
connections = connections.groupby(['writer', 'receiver'], as_index=False).sum()  # Group by writer and receiver, and sum the weights

connections

In [ ]:
# Create an undirected graph
G = nx.Graph()

# Add edges using a loop
for _, row in connections.iterrows():
    G.add_edge(row['writer'], row['receiver'], weight=row['weight'])

# Use spring layout to position nodes
pos = nx.spring_layout(G, k=0.3, iterations=50)

# Visualize the graph
plt.figure(figsize=(20, 20))
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1000, font_size=8, font_weight='bold', edge_color='gray')
plt.title('Letter Exchange Network (Full, Spring Layout)')
plt.savefig("ming_letter_exchange_network_full_spring.png", format="png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Create an undirected graph
G = nx.Graph()

# Add edges using a loop
for _, row in connections.iterrows():
    G.add_edge(row['writer'], row['receiver'], weight=row['weight'])

# Use spring layout to position nodes
pos = nx.forceatlas2_layout(G)

# Visualize the graph
plt.figure(figsize=(20, 20))
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1000, font_size=8, font_weight='bold', edge_color='gray')
plt.title('Letter Exchange Network (Full, Force Atlas 2)')
plt.savefig("ming_letter_exchange_network_full_forceatlas2.png", format="png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Count the number of letters written by each person
writer_counts = connections['writer'].value_counts()

# Keep only writers who wrote to 100 or more people
valid_writers = writer_counts[writer_counts >= 100].index

connections_filtered = connections[connections['writer'].isin(valid_writers)]
print(len(connections_filtered))

# Create an undirected graph
G = nx.Graph()

# Add edges using a loop
for _, row in connections_filtered.iterrows():
    G.add_edge(row['writer'], row['receiver'], weight=row['weight'])

pos = nx.forceatlas2_layout(G)

# Visualize the graph
plt.figure(figsize=(20, 20))
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1000, font_size=8, font_weight='bold', edge_color='gray')
plt.title('Letter Exchange Network (Filtered, Force Atlas 2)')
plt.savefig("ming_letter_exchange_network_filtered_forceatlas2.png", format="png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Find nodes with degree 1
nodes_with_degree_one = [node for node, degree in G.degree() if degree == 1]

# Find the "bridge" node they are connected to
bridge_connections = {}
for node in nodes_with_degree_one:
    neighbors = list(G.neighbors(node))
    if neighbors:  # Ensure there is a neighbor
        bridge_node = neighbors[0]
        if bridge_node not in bridge_connections:
            bridge_connections[bridge_node] = []
        bridge_connections[bridge_node].append(node)

# Print bridge nodes and their connected nodes
for bridge_node, connected_nodes in bridge_connections.items():
    if len(connected_nodes) > 1:  # Only consider if there are multiple nodes connected
        print(f"Bridge Node: {bridge_node}, Connected Nodes: {connected_nodes}")


In [ ]:
evc = nx.eigenvector_centrality(G)
sorted(((node, centrality) for node, centrality in evc.items()), key = lambda x:-x[1])[:100]

In [ ]:
import networkx as nx
from pyvis.network import Network
from community import community_louvain
import numpy as np

# Assuming you have the `connections_filtered` DataFrame
G = nx.Graph()

# Add edges using a loop
for _, row in connections_filtered.iterrows():
    G.add_edge(row['writer'], row['receiver'], weight=row['weight'])

# Initialize Pyvis network
net = Network(notebook=True, cdn_resources='in_line', width="1000px", height="1000px", bgcolor='#222222', font_color='white')

# Setting the pos attribute
pos = nx.forceatlas2_layout(G)
scaling_pos = 10
pos_list = {node: [float(coord * scaling_pos) for coord in coords] for node, coords in pos.items()}
nx.set_node_attributes(G, pos_list, 'pos')

# Setting the size attribute
evc = nx.eigenvector_centrality(G)
scaling_size = 300
node_centrality = {node: size * scaling_size for node, size in evc.items()}  # Rescale sizes
nx.set_node_attributes(G, node_centrality, 'size')

# Setting the community attribute
communities = community_louvain.best_partition(G)
nx.set_node_attributes(G, communities, 'group')

# Add nodes and edges to Pyvis network, retaining attributes
net.from_nx(G)

# Update node positions manually
for node in net.nodes:
    if 'pos' in G.nodes[node['id']]:
        x, y = G.nodes[node['id']]['pos']
        node['x'] = x
        node['y'] = y

    node['font'] = {'size': 30, 'color': 'white'}

# Disable physics to keep positions static
net.toggle_physics(False)

# Display the network
net.show("ming_letters.html")

In [ ]:
import pandas as pd

# Assuming `connections` is your DataFrame with columns: 'writer', 'receiver', and 'weight'
# Create a set of unique nodes from the connections
unique_nodes = set(connections['writer']).union(set(connections['receiver']))

# Create a DataFrame for nodes
nodes_df = pd.DataFrame(list(unique_nodes), columns=['Id'])

# Create a DataFrame for edges (connections)
edges_df = connections.rename(columns={'writer': 'Source', 'receiver': 'Target', 'weight': 'Weight'})

# Save nodes and edges to CSV files
nodes_df.to_csv("nodes.csv", index=False, encoding='utf-8')
edges_df.to_csv("edges.csv", index=False, encoding='utf=8')

print("Files 'nodes.csv' and 'edges.csv' have been saved.")